# Stage 2 Training: Cascaded Deep Spatial Refiner (10m → 2.5m → 1.25m)
### B.Tech Major Project: Ultra-High Resolution Satellite Mapping (~1 Meter GSD)

This notebook executes the training for the **Stage 2 Cascaded Deep Spatial Refiner**:
- **Stage 1 (Frozen Backbone):** Fine-Tuned 4× SEN2SR Lite (`sen2sr_lite_compound_finetuned.pth`) upscaling Sentinel-2 from **10m → 2.5m** with radiometric and multispectral index preservation.
- **Stage 2 (Trained Refiner):** 2× Deep Residual Attention Network upscaling Stage 1 outputs from **2.5m → 1.25m** (~1 meter ground resolution) against native sub-meter NAIP aerial ground truth.

**Key Innovations:**
1. **Progressive / Cascaded SR:** Bridges the 8× resolution gap in two stable stages ($4\times \times 2\times = 8\times$).
2. **Multi-Scale Compound Loss:** Supervised with Pixel $L_1$ + Sobel Gradient Edge Loss + Differentiable NDVI/NDWI Spectral Fidelity.
3. **RAM-Cached Acceleration:** Full dataset cached in RAM on Epoch 1; Epochs 2–30 execute at maximum GPU speed (~12 minutes total on Colab T4).

## Step 1 — Mount Google Drive & Environment Configuration

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, sys, glob, time, warnings, logging, shutil

# Silence GDAL & rasterio CPL messages
os.environ['CPL_LOG_ERRORS'] = 'OFF'
os.environ['CPL_LOG'] = 'OFF'
os.environ['GDAL_PAM_ENABLED'] = 'NO'
warnings.filterwarnings('ignore')
for l_name in ['rasterio', 'rasterio._env', 'rasterio.env', 'rasterio._base']:
    l = logging.getLogger(l_name)
    l.setLevel(logging.ERROR)
    l.propagate = False

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from tqdm.auto import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

BASE_DIR = '/content/drive/MyDrive/Project_Folder'
DATASETS_DIR = os.path.join(BASE_DIR, 'Datasets')
SEN2NAIP_LR = os.path.join(DATASETS_DIR, 'sentinaiplow')
SEN2NAIP_HR = os.path.join(DATASETS_DIR, 'sentinaiphigh')
OUTPUT_DIR = os.path.join(BASE_DIR, 'Results')
os.makedirs(OUTPUT_DIR, exist_ok=True)

STAGE1_CKPT = os.path.join(OUTPUT_DIR, 'sen2sr_lite_compound_finetuned.pth')
print(f'Stage 1 Fine-Tuned Checkpoint exists: {os.path.exists(STAGE1_CKPT)}')
if not os.path.exists(STAGE1_CKPT):
    raise FileNotFoundError(f'Stage 1 checkpoint not found at: {STAGE1_CKPT}')

## Step 2 — Install Required Libraries

In [ ]:
%pip install -q sen2sr mlstac rasterio safetensors pandas matplotlib

## Step 3 — Define Stage 2 Deep Residual Refiner Architecture (2×)

In [ ]:
class ResidualDenseBlock(nn.Module):
    def __init__(self, channels=48, growth_rate=24):
        super().__init__()
        self.c1 = nn.Conv2d(channels, growth_rate, 3, padding=1)
        self.c2 = nn.Conv2d(channels + growth_rate, growth_rate, 3, padding=1)
        self.c3 = nn.Conv2d(channels + 2 * growth_rate, channels, 3, padding=1)
        self.act = nn.LeakyReLU(0.05, inplace=True)

    def forward(self, x):
        x1 = self.act(self.c1(x))
        x2 = self.act(self.c2(torch.cat([x, x1], 1)))
        x3 = self.c3(torch.cat([x, x1, x2], 1))
        return x + (0.2 * x3)


class Stage2SpatialRefiner(nn.Module):
    """
    2x Deep Spatial Refinement Network upscaling 2.5m RGBN inputs to ~1.25m ultra-resolution.
    """
    def __init__(self, in_channels=4, out_channels=4, feat_channels=48, num_blocks=5):
        super().__init__()
        self.head = nn.Conv2d(in_channels, feat_channels, 3, padding=1)
        self.blocks = nn.ModuleList([ResidualDenseBlock(feat_channels) for _ in range(num_blocks)])
        self.body_conv = nn.Conv2d(feat_channels, feat_channels, 3, padding=1)
        # 2x PixelShuffle Upsampler (feat_channels -> out_channels * 4 -> out_channels)
        self.upsampler = nn.Sequential(
            nn.Conv2d(feat_channels, out_channels * 4, 3, padding=1),
            nn.PixelShuffle(2)
        )
        # Residual skip from interpolated input
        self.skip_interp = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)

    def forward(self, x):
        f_in = self.head(x)
        f = f_in
        for b in self.blocks:
            f = b(f)
        f = f_in + self.body_conv(f)
        sr = self.upsampler(f)
        skip = self.skip_interp(x)
        return torch.clamp(sr + skip, 0.0, 1.0)

model_refiner = Stage2SpatialRefiner(in_channels=4, out_channels=4, feat_channels=48, num_blocks=5).to(device)
n_params = sum(p.numel() for p in model_refiner.parameters())
print(f'Stage 2 Deep Refiner Architecture instantiated ({n_params:,} parameters).')

## Step 4 — Multi-Objective Compound Loss (Spatial + Spectral)

In [ ]:
class EdgeAwareSobelLoss(nn.Module):
    def __init__(self):
        super().__init__()
        sx = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=torch.float32).view(1, 1, 3, 3)
        sy = torch.tensor([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=torch.float32).view(1, 1, 3, 3)
        self.register_buffer('sx', sx)
        self.register_buffer('sy', sy)

    def forward(self, pred, target):
        loss = 0.0
        for c in range(pred.shape[1]):
            p = pred[:, c:c+1]
            t = target[:, c:c+1]
            gx_p = F.conv2d(p, self.sx, padding=1)
            gy_p = F.conv2d(p, self.sy, padding=1)
            gx_t = F.conv2d(t, self.sx, padding=1)
            gy_t = F.conv2d(t, self.sy, padding=1)
            loss += F.l1_loss(gx_p, gx_t) + F.l1_loss(gy_p, gy_t)
        return loss / pred.shape[1]


class MultiIndexSpectralLoss(nn.Module):
    def __init__(self, eps=1e-4, weight_ndwi=0.5):
        super().__init__()
        self.eps = eps
        self.weight_ndwi = weight_ndwi

    def forward(self, pred, target):
        p_c = torch.clamp(pred, 0.0, 1.0)
        t_c = torch.clamp(target, 0.0, 1.0)
        # 0:Blue, 1:Green, 2:Red, 3:NIR
        g_p, g_t = p_c[:, 1], t_c[:, 1]
        r_p, r_t = p_c[:, 2], t_c[:, 2]
        nir_p, nir_t = p_c[:, 3], t_c[:, 3]

        # NDVI (Vegetation)
        ndvi_p = (nir_p - r_p) / (nir_p + r_p).clamp(min=self.eps)
        ndvi_t = (nir_t - r_t) / (nir_t + r_t).clamp(min=self.eps)
        l_ndvi = F.l1_loss(ndvi_p, ndvi_t)

        # NDWI (Water / Moisture)
        ndwi_p = (g_p - nir_p) / (g_p + nir_p).clamp(min=self.eps)
        ndwi_t = (g_t - nir_t) / (g_t + nir_t).clamp(min=self.eps)
        l_ndwi = F.l1_loss(ndwi_p, ndwi_t)

        return l_ndvi + (self.weight_ndwi * l_ndwi), {'ndvi': l_ndvi.item(), 'ndwi': l_ndwi.item()}


class CascadedRefinementLoss(nn.Module):
    def __init__(self, lambda_edge=0.15, lambda_spec=0.10):
        super().__init__()
        self.l1 = nn.L1Loss()
        self.edge = EdgeAwareSobelLoss()
        self.spec = MultiIndexSpectralLoss()
        self.lambda_edge = lambda_edge
        self.lambda_spec = lambda_spec

    def forward(self, pred, target):
        loss_l1 = self.l1(pred, target)
        loss_edge = self.edge(pred, target)
        loss_spec, spec_info = self.spec(pred, target)
        total = loss_l1 + (self.lambda_edge * loss_edge) + (self.lambda_spec * loss_spec)
        return total, {'l1': loss_l1.item(), 'edge': loss_edge.item(), **spec_info}

## Step 5 — Load Frozen Stage 1 Model & Build Paired 2.5m → 1.25m Dataset

In [ ]:
import rasterio
from sen2sr.models.opensr_baseline.cnn import CNNSR

# 1. Load Frozen Stage 1 Model (10m -> 2.5m)
stage1_model = CNNSR(in_channels=4, out_channels=4, feature_channels=24, upscale=4, bias=True, train_mode=True, num_blocks=6)
stage1_weights = torch.load(STAGE1_CKPT, map_location=device)
stage1_model.load_state_dict(stage1_weights)
stage1_model = stage1_model.to(device).eval()
for p in stage1_model.parameters():
    p.requires_grad = False
print('Stage 1 Fine-Tuned Generator (10m -> 2.5m) loaded and frozen.')

# 2. Dataset providing Stage 1 2.5m outputs paired with native ~1.25m NAIP ground truth
class CascadedRefinementDataset(Dataset):
    def __init__(self, lr_dir, hr_dir, stage1_model, max_samples=400, crop_lr=32):
        lr_files = sorted(glob.glob(os.path.join(lr_dir, '*.tif')))
        hr_files = sorted(glob.glob(os.path.join(hr_dir, '*.tif')))
        hr_map = {os.path.basename(f).replace('_hr.tif', ''): f for f in hr_files}
        self.pairs = []
        for f in lr_files:
            k = os.path.basename(f).replace('_lr.tif', '')
            if k in hr_map:
                self.pairs.append((f, hr_map[k]))
        if max_samples:
            self.pairs = self.pairs[:max_samples]
        
        self.crop_lr = crop_lr
        self.stage1_model = stage1_model
        self.cache = {}
        print(f'Matched {len(self.pairs)} paired scenes for Stage 2 Refinement.')

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        if idx in self.cache:
            return self.cache[idx]
        
        lr_path, hr_path = self.pairs[idx]
        with rasterio.open(lr_path) as s:
            lr_raw = s.read().astype(np.float32)
        with rasterio.open(hr_path) as s:
            hr_raw = s.read().astype(np.float32)
            
        # Normalize reflectance
        lr_norm = lr_raw / 10000.0 if np.nanmax(lr_raw) > 255.0 else (lr_raw / 255.0 if np.nanmax(lr_raw) > 2.0 else lr_raw)
        hr_norm = hr_raw / 10000.0 if np.nanmax(hr_raw) > 255.0 else (hr_raw / 255.0 if np.nanmax(hr_raw) > 2.0 else hr_raw)
        
        lr_chw = np.nan_to_num(lr_norm[:4], nan=0.0, posinf=1.0, neginf=0.0)
        hr_chw = np.nan_to_num(hr_norm[:4], nan=0.0, posinf=1.0, neginf=0.0)
        
        # 8x spatial target: ensure HR is strictly 8x LR spatial dimensions
        _, h_lr, w_lr = lr_chw.shape
        hr_target_h, hr_target_w = h_lr * 8, w_lr * 8
        hr_t = torch.from_numpy(hr_chw).unsqueeze(0).float()
        hr_8x = F.interpolate(hr_t, size=(hr_target_h, hr_target_w), mode='bilinear', align_corners=False).squeeze(0)
        
        # Synchronized random crop
        c_lr = self.crop_lr
        y = np.random.randint(0, max(1, h_lr - c_lr))
        x = np.random.randint(0, max(1, w_lr - c_lr))
        
        patch_lr = torch.from_numpy(lr_chw[:, y:y+c_lr, x:x+c_lr]).unsqueeze(0).float().to(device)
        patch_hr_target = hr_8x[:, y*8:(y+c_lr)*8, x*8:(x+c_lr)*8].float()
        
        # Run Stage 1 model to generate 2.5m intermediate tensor
        with torch.no_grad():
            patch_2dot5m = self.stage1_model(patch_lr).squeeze(0).cpu()
            
        item = (patch_2dot5m, patch_hr_target)
        self.cache[idx] = item
        return item

full_dataset = CascadedRefinementDataset(SEN2NAIP_LR, SEN2NAIP_HR, stage1_model, max_samples=400, crop_lr=32)
train_size = int(0.85 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_set, val_set = random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_set, batch_size=8, shuffle=True, drop_last=True)
val_loader = DataLoader(val_set, batch_size=8, shuffle=False)
print(f'Train Batches: {len(train_loader)} | Val Batches: {len(val_loader)}')

## Step 6 — Execute Extended 30-Epoch Training Loop with Cosine Decay

In [ ]:
criterion = CascadedRefinementLoss(lambda_edge=0.15, lambda_spec=0.10).to(device)
optimizer = torch.optim.AdamW(model_refiner.parameters(), lr=1.5e-4, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30, eta_min=1e-6)

EPOCHS = 30
history = {'train_loss': [], 'val_loss': [], 'train_l1': [], 'ndvi_loss': [], 'ndwi_loss': []}

print('Starting Stage 2 Spatial Refiner Training (30 Epochs, ~1.25m Ultra-Resolution)...')
t0 = time.time()

for epoch in range(1, EPOCHS + 1):
    model_refiner.train()
    tr_loss, tr_l1, tr_ndvi, tr_ndwi = 0.0, 0.0, 0.0, 0.0
    
    for in_2dot5m, hr_target in train_loader:
        in_2dot5m, hr_target = in_2dot5m.to(device), hr_target.to(device)
        optimizer.zero_grad()
        pred_1m = model_refiner(in_2dot5m)
        loss, info = criterion(pred_1m, hr_target)
        loss.backward()
        optimizer.step()
        
        tr_loss += loss.item()
        tr_l1 += info['l1']
        tr_ndvi += info['ndvi']
        tr_ndwi += info['ndwi']
        
    scheduler.step()
    n_b = len(train_loader)
    tr_loss /= n_b
    tr_l1 /= n_b
    tr_ndvi /= n_b
    tr_ndwi /= n_b
    
    # Validation
    model_refiner.eval()
    v_loss = 0.0
    with torch.no_grad():
        for in_2dot5m, hr_target in val_loader:
            in_2dot5m, hr_target = in_2dot5m.to(device), hr_target.to(device)
            pred_1m = model_refiner(in_2dot5m)
            loss, _ = criterion(pred_1m, hr_target)
            v_loss += loss.item()
    v_loss /= len(val_loader)
    
    history['train_loss'].append(tr_loss)
    history['val_loss'].append(v_loss)
    history['train_l1'].append(tr_l1)
    history['ndvi_loss'].append(tr_ndvi)
    history['ndwi_loss'].append(tr_ndwi)
    
    if epoch % 5 == 0 or epoch == 1 or epoch == EPOCHS:
        print(f'Epoch [{epoch:02d}/{EPOCHS:02d}] - Total: {tr_loss:.4f} | Val: {v_loss:.4f} | L1: {tr_l1:.4f} | NDVI: {tr_ndvi:.4f} | NDWI: {tr_ndwi:.4f}')

elapsed_mins = (time.time() - t0) / 60.0
print(f'\nStage 2 Training Completed in {elapsed_mins:.2f} minutes!')

# Save Checkpoint
STAGE2_CKPT = os.path.join(OUTPUT_DIR, 'sen2sr_cascaded_1meter_refiner.pth')
torch.save(model_refiner.state_dict(), STAGE2_CKPT)
print(f'Saved Stage 2 Refiner checkpoint to: {STAGE2_CKPT}')

## Step 7 — Validation Convergence Curves & Visual Super-Resolution Benchmark

In [ ]:
# 1. Plot Loss Curves
plt.figure(figsize=(12, 5))
plt.plot(history['train_loss'], label='Train Total Compound Loss', color='#1976D2', lw=2)
plt.plot(history['val_loss'], label='Validation Total Loss', color='#D32F2F', lw=2, linestyle='--')
plt.plot(history['train_l1'], label='Pixel L1 Reconstruction Error', color='#388E3C', lw=1.5)
plt.title('Stage 2 Spatial Refiner: 30-Epoch Convergence (~1.25m)', fontsize=12, fontweight='bold')
plt.xlabel('Epoch', fontsize=11)
plt.ylabel('Loss Value', fontsize=11)
plt.legend(frameon=True)
plt.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
curve_path = os.path.join(OUTPUT_DIR, 'SEN2SR_1Meter_Training_Loss_Curve.png')
plt.savefig(curve_path, dpi=200, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Training curve saved to: {curve_path}')

# 2. Validation Sample Visual Super-Resolution
model_refiner.eval()
with torch.no_grad():
    in_2dot5m, hr_target = next(iter(val_loader))
    in_2dot5m, hr_target = in_2dot5m.to(device), hr_target.to(device)
    pred_1m = model_refiner(in_2dot5m)

def to_rgb(t):
    # B04 (Red: 2), B03 (Green: 1), B02 (Blue: 0)
    rgb = t[[2, 1, 0]].permute(1, 2, 0).cpu().numpy()
    lo, hi = np.percentile(rgb, [2, 98])
    return np.clip((rgb - lo) / max(hi - lo, 1e-6), 0, 1)

fig, axes = plt.subplots(1, 3, figsize=(15, 6))
axes[0].imshow(to_rgb(in_2dot5m[0]))
axes[0].set_title('Stage 1 Output: 2.5m (Fine-Tuned SEN2SR)', fontsize=11, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(to_rgb(pred_1m[0]))
axes[1].set_title('Stage 2 Output: 1.25m (Cascaded Refiner, Ours)', fontsize=11, fontweight='bold')
axes[1].axis('off')

axes[2].imshow(to_rgb(hr_target[0]))
axes[2].set_title('Aerial Ground Truth (~1m NAIP)', fontsize=11, fontweight='bold')
axes[2].axis('off')

plt.tight_layout()
val_save_path = os.path.join(OUTPUT_DIR, 'SEN2SR_1Meter_Validation_Comparison.png')
plt.savefig(val_save_path, dpi=200, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Validation comparison saved to: {val_save_path}')